# Part 2 - Bias Audit

Compute TPR/FPR/FNR/precision by cohort and highlight disparity.

In [ ]:
import os
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from assignment_workflow import load_and_split_data, tokenize_df, cohort_metrics, plot_grouped_rates
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

DATA_PATH = "/content/drive/MyDrive/jigsaw-unintended-bias-train.csv"
print("Data exists:", os.path.exists(DATA_PATH))

_, eval_df = load_and_split_data(DATA_PATH)
model_dir = 'saved_model/part1_baseline'
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)
trainer = Trainer(model=model, args=TrainingArguments(output_dir='tmp_part2', report_to=[]))

eval_dataset = tokenize_df(eval_df, tokenizer)
logits = trainer.predict(eval_dataset).predictions
import torch, numpy as np
y_prob = torch.softmax(torch.tensor(logits), dim=1).numpy()[:,1]

In [ ]:
m = cohort_metrics(eval_df, y_prob, threshold=0.5)
print(m)
plot_grouped_rates(m)

for cohort_name, mask in [('high_black', eval_df['black'] >= 0.5), ('reference', (eval_df['black'] < 0.1) & (eval_df['white'] >= 0.5))]:
    y_true = eval_df.loc[mask, 'label'].values
    y_pred = (y_prob[mask.values] >= 0.5).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0,1])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix - {cohort_name}')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()